# T37 - 同花顺Excel导入财务指标

## 7. 完整执行流程

本章展示完整的财务指标导入执行流程。

## 7.1 环境准备

In [ ]:
# 导入标准库
import os
import sys
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path
from time import sleep

# 添加项目路径
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

# 导入第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
import psycopg2

# 导入项目配置
from config import (
    PROJECT_NAME,
    PROJECT_VERSION,
    get_mysql_bond_engine,
    get_postgres_engine,
    get_postgres_connection,
    get_ths_account,
    POSTGRES_URL,
    CHUNK_SIZE,
    TABLE_WIDE_FORMAT,
    TABLE_LONG_FORMAT
)

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"项目: {PROJECT_NAME}")
print(f"版本: {PROJECT_VERSION}")
print("=" * 50)

## 7.2 执行流程概览

```
┌─────────────────────────────────────────────────────────────┐
│  步骤1：数据准备                                             │
├─────────────────────────────────────────────────────────────┤
│  - 验证数据库连接                                           │
│  - 验证同花顺API登录                                        │
│  - 查询发行人代码列表                                       │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  步骤2：同花顺API批量提取                                    │
├─────────────────────────────────────────────────────────────┤
│  - 循环日期列表（20230630, 20221231, ...）                   │
│  - 调用THS_BD提取150+个财务指标                             │
│  - 写入PostgreSQL宽表                                       │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  步骤3：Excel批量处理（可选）                                │
├─────────────────────────────────────────────────────────────┤
│  - 填充日期列                                               │
│  - 执行公式计算                                             │
│  - 保存为值                                                 │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  步骤4：数据导入与转换                                       │
├─────────────────────────────────────────────────────────────┤
│  - Excel批量导入PostgreSQL                                  │
│  - 列类型转换为FLOAT                                        │
│  - 宽表转长表                                               │
└─────────────────────────────────────────────────────────────┘
```

## 7.3 步骤1：数据准备

In [ ]:
class FinancialDataPipeline:
    """财务数据处理管道"""
    
    def __init__(self):
        """初始化管道"""
        self.mysql_engine = None
        self.pg_engine = None
        self.pg_conn = None
        self.ths_logged_in = False
        
    def setup(self):
        """
        初始化所有连接
        
        Returns:
        --------
        bool : 初始化是否成功
        """
        logger.info("=" * 50)
        logger.info("开始初始化数据管道")
        logger.info("=" * 50)
        
        # 1. 初始化MySQL连接
        try:
            self.mysql_engine = get_mysql_bond_engine()
            with self.mysql_engine.connect() as conn:
                conn.execute(text("SELECT 1"))
            logger.info("MySQL连接成功")
        except Exception as e:
            logger.error(f"MySQL连接失败: {e}")
            return False
            
        # 2. 初始化PostgreSQL连接
        try:
            self.pg_engine = get_postgres_engine()
            self.pg_conn = get_postgres_connection()
            with self.pg_engine.connect() as conn:
                conn.execute(text("SELECT 1"))
            logger.info("PostgreSQL连接成功")
        except Exception as e:
            logger.error(f"PostgreSQL连接失败: {e}")
            return False
            
        # 3. 登录同花顺API（可选）
        try:
            from iFinDPy import THS_iFinDLogin
            account = get_ths_account()
            if account:
                result = THS_iFinDLogin(account['username'], account['password'])
                if result == 0:
                    self.ths_logged_in = True
                    logger.info("同花顺API登录成功")
                else:
                    logger.warning(f"同花顺API登录失败，错误码: {result}")
            else:
                logger.warning("未配置同花顺API账号")
        except ImportError:
            logger.warning("iFinDPy模块未安装，跳过同花顺API登录")
        except Exception as e:
            logger.warning(f"同花顺API登录异常: {e}")
            
        logger.info("数据管道初始化完成")
        return True
    
    def get_issuers(self):
        """
        获取发行人代码列表
        
        Returns:
        --------
        list : 发行人代码列表
        """
        sql = """
        SELECT 
            max(trade_code) AS trade_code,
            ths_issuer_name_cn_bond
        FROM (
            SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_credit
            UNION ALL
            SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_finance
            UNION ALL
            SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_abs
        ) SQ
        GROUP BY ths_issuer_name_cn_bond
        """
        
        df = pd.read_sql(sql, self.pg_engine)
        trade_codes = df['trade_code'].tolist()
        logger.info(f"获取到 {len(trade_codes)} 个发行人代码")
        return trade_codes

# 创建数据管道
pipeline = FinancialDataPipeline()
setup_success = pipeline.setup()
print(f"\n初始化结果: {'成功' if setup_success else '失败'}")

## 7.4 步骤2：同花顺API批量提取

In [ ]:
def run_ths_extraction(pipeline, dates, table_name=TABLE_WIDE_FORMAT):
    """
    执行同花顺API批量提取
    
    Parameters:
    -----------
    pipeline : FinancialDataPipeline
        数据管道
    dates : list
        日期列表 [(dt, dt1), ...]
    table_name : str
        目标表名
        
    Returns:
    --------
    dict : 执行统计
    """
    if not pipeline.ths_logged_in:
        logger.warning("同花顺API未登录，跳过提取")
        return {'status': 'skipped', 'reason': 'API未登录'}
        
    logger.info("=" * 50)
    logger.info("开始同花顺API批量提取")
    logger.info("=" * 50)
    
    stats = {
        'status': 'success',
        'total_issuers': 0,
        'total_extracted': 0,
        'errors': []
    }
    
    # 获取发行人代码
    trade_codes = pipeline.get_issuers()
    stats['total_issuers'] = len(trade_codes)
    
    try:
        from iFinDPy import THS_BD
        
        for dt, dt1 in dates:
            logger.info(f"处理报告期: {dt}")
            
            for i, trade_code in enumerate(trade_codes):
                try:
                    # 构建指标和参数（简化版）
                    indicators = 'ths_total_assets_bond;ths_total_liab_bond;ths_currency_fund_bond'
                    params = f'{dt},100;{dt};{dt}'
                    
                    # 调用API
                    df = THS_BD(trade_code, indicators, params)
                    
                    if df is not None and df.data is not None:
                        result = df.data
                        result['dt'] = dt
                        result.dropna(how='all', inplace=True)
                        
                        # 保存到数据库
                        result.to_sql(table_name, POSTGRES_URL, if_exists='append', index=False)
                        stats['total_extracted'] += 1
                        
                    # 进度显示
                    if (i + 1) % 100 == 0:
                        logger.info(f"已处理 {i+1}/{len(trade_codes)} 个发行人")
                        
                    # 避免API频率限制
                    sleep(0.3)
                    
                except Exception as e:
                    stats['errors'].append(f"{trade_code}: {str(e)}")
                    
    except ImportError:
        stats['status'] = 'failed'
        stats['reason'] = 'iFinDPy模块未安装'
    except Exception as e:
        stats['status'] = 'failed'
        stats['reason'] = str(e)
        
    logger.info(f"提取完成: {stats['total_extracted']}/{stats['total_issuers']}")
    return stats

# 定义日期列表
extraction_dates = [
    ('20230630', '2023-06-30'),
    ('20221231', '2022-12-31')
]

# 执行提取（需要API登录）
# extraction_stats = run_ths_extraction(pipeline, extraction_dates)

## 7.5 步骤3：Excel批量处理

In [ ]:
def run_excel_processing(folder_path):
    """
    执行Excel批量处理
    
    Parameters:
    -----------
    folder_path : str
        Excel文件夹路径
        
    Returns:
    --------
    dict : 执行统计
    """
    logger.info("=" * 50)
    logger.info("开始Excel批量处理")
    logger.info("=" * 50)
    
    stats = {
        'status': 'success',
        'total_files': 0,
        'processed_files': 0,
        'errors': []
    }
    
    if not os.path.exists(folder_path):
        logger.warning(f"文件夹不存在: {folder_path}")
        return {'status': 'skipped', 'reason': '文件夹不存在'}
        
    files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]
    stats['total_files'] = len(files)
    
    if len(files) == 0:
        logger.warning("未找到Excel文件")
        return {'status': 'skipped', 'reason': '未找到Excel文件'}
        
    try:
        import win32com.client as win32
        
        excel = win32.gencache.EnsureDispatch('Excel.Application')
        excel.DisplayAlerts = False
        
        for filename in files:
            file_path = os.path.join(folder_path, filename)
            
            try:
                wb = excel.Workbooks.Open(file_path)
                ws = wb.Worksheets(1)
                
                # 提取日期
                date_str = filename.split('.')[0]
                date_obj = datetime.strptime(date_str, '%Y%m%d')
                formatted_date = date_obj.strftime('%Y-%m-%d')
                
                # 填充日期列
                last_column = ws.Cells(1, ws.Columns.Count).End(win32.constants.xlToLeft).Column
                ws.Range(
                    ws.Cells(2, last_column),
                    ws.Cells(ws.UsedRange.Rows.Count, last_column)
                ).Value = formatted_date
                
                # 自动填充
                col = last_column - 1
                source_range = ws.Range(ws.Cells(2, col), ws.Cells(2, col))
                destination_range = ws.Range(
                    ws.Cells(2, col),
                    ws.Cells(ws.UsedRange.Rows.Count, col)
                )
                source_range.AutoFill(destination_range, win32.constants.xlFillDefault)
                
                wb.Save()
                wb.Close()
                
                stats['processed_files'] += 1
                logger.info(f"处理 {filename} 完成")
                
            except Exception as e:
                stats['errors'].append(f"{filename}: {str(e)}")
                logger.error(f"处理 {filename} 失败: {e}")
                
        excel.Quit()
        
    except ImportError:
        stats['status'] = 'failed'
        stats['reason'] = 'win32com模块未安装'
    except Exception as e:
        stats['status'] = 'failed'
        stats['reason'] = str(e)
        
    logger.info(f"Excel处理完成: {stats['processed_files']}/{stats['total_files']}")
    return stats

# 执行Excel处理（需要Windows环境）
# excel_stats = run_excel_processing('C:\\Users\\Administrator\\Desktop\\新建文件夹')

## 7.6 步骤4：数据导入与转换

In [ ]:
def run_data_import(folder_path, wide_table=TABLE_WIDE_FORMAT, long_table=TABLE_LONG_FORMAT):
    """
    执行数据导入与转换
    
    Parameters:
    -----------
    folder_path : str
        Excel文件夹路径
    wide_table : str
        宽表名
    long_table : str
        长表名
        
    Returns:
    --------
    dict : 执行统计
    """
    logger.info("=" * 50)
    logger.info("开始数据导入与转换")
    logger.info("=" * 50)
    
    stats = {
        'status': 'success',
        'total_rows_imported': 0,
        'total_rows_converted': 0,
        'errors': []
    }
    
    # 1. 批量导入Excel
    if os.path.exists(folder_path):
        files = [f for f in os.listdir(folder_path) 
                 if f.endswith('.xlsx') or f.endswith('.xls')]
        
        for filename in files:
            file_path = os.path.join(folder_path, filename)
            
            try:
                df = pd.read_excel(file_path)
                df.to_sql(wide_table, POSTGRES_URL, if_exists='append', index=False)
                stats['total_rows_imported'] += len(df)
                logger.info(f"导入 {filename}: {len(df)} 行")
                
            except Exception as e:
                stats['errors'].append(f"导入{filename}: {str(e)}")
                
        logger.info(f"总计导入 {stats['total_rows_imported']} 行")
        
    # 2. 列类型转换
    logger.info("开始列类型转换...")
    try:
        pg_engine = get_postgres_engine()
        
        # 获取列名
        sql = f"""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = '{wide_table}'
        """
        df_cols = pd.read_sql(sql, pg_engine)
        columns = [c for c in df_cols['column_name'].tolist() 
                   if c not in ['dt', 'thscode']]
        
        # 转换类型
        pg_conn = get_postgres_connection()
        cursor = pg_conn.cursor()
        
        for col in columns:
            try:
                cursor.execute(f"""
                    ALTER TABLE {wide_table}
                    ALTER COLUMN {col} TYPE FLOAT USING {col}::FLOAT
                """)
            except Exception as e:
                logger.warning(f"转换列 {col} 失败: {e}")
                
        pg_conn.commit()
        cursor.close()
        pg_conn.close()
        logger.info("列类型转换完成")
        
    except Exception as e:
        stats['errors'].append(f"列类型转换: {str(e)}")
        
    # 3. 宽表转长表
    logger.info("开始宽表转长表...")
    try:
        query = f"SELECT * FROM {wide_table}"
        chunks = pd.read_sql(query, POSTGRES_URL, chunksize=CHUNK_SIZE)
        
        for df in chunks:
            df_long = df.melt(
                id_vars=['dt', 'thscode'],
                var_name='指标',
                value_name='value'
            )
            df_long.rename(columns={'thscode': 'trade_code'}, inplace=True)
            
            df_long.to_sql(long_table, POSTGRES_URL, if_exists='append', index=False)
            stats['total_rows_converted'] += len(df)
            
        logger.info(f"转换完成: {stats['total_rows_converted']} 行")
        
    except Exception as e:
        stats['errors'].append(f"宽表转长表: {str(e)}")
        
    return stats

# 执行数据导入
# import_stats = run_data_import('C:\\Users\\Administrator\\Desktop\\新建文件夹\\1')

## 7.7 完整执行流程

In [ ]:
def run_full_pipeline(
    excel_folder=None,
    dates=None,
    enable_ths_extraction=True,
    enable_excel_processing=True,
    enable_data_import=True
):
    """
    执行完整的数据处理管道
    
    Parameters:
    -----------
    excel_folder : str
        Excel文件夹路径
    dates : list
        提取日期列表
    enable_ths_extraction : bool
        是否启用API提取
    enable_excel_processing : bool
        是否启用Excel处理
    enable_data_import : bool
        是否启用数据导入
        
    Returns:
    --------
    dict : 执行报告
    """
    start_time = datetime.now()
    
    report = {
        'start_time': start_time.isoformat(),
        'pipeline_status': 'success',
        'steps': {}
    }
    
    # 初始化
    logger.info("=" * 60)
    logger.info("开始执行财务指标导入完整流程")
    logger.info("=" * 60)
    
    pipeline = FinancialDataPipeline()
    if not pipeline.setup():
        report['pipeline_status'] = 'failed'
        report['error'] = '初始化失败'
        return report
        
    # 步骤1：同花顺API提取
    if enable_ths_extraction and dates:
        report['steps']['ths_extraction'] = run_ths_extraction(pipeline, dates)
        
    # 步骤2：Excel处理
    if enable_excel_processing and excel_folder:
        report['steps']['excel_processing'] = run_excel_processing(excel_folder)
        
    # 步骤3：数据导入
    if enable_data_import and excel_folder:
        report['steps']['data_import'] = run_data_import(excel_folder)
        
    # 完成报告
    end_time = datetime.now()
    report['end_time'] = end_time.isoformat()
    report['duration_seconds'] = (end_time - start_time).total_seconds()
    
    logger.info("=" * 60)
    logger.info(f"流程完成，耗时: {report['duration_seconds']:.2f} 秒")
    logger.info("=" * 60)
    
    return report

# 执行完整流程示例
full_report = run_full_pipeline(
    excel_folder=None,  # 设置实际的Excel文件夹路径
    dates=[('20230630', '2023-06-30')],
    enable_ths_extraction=False,  # 需要API登录时设为True
    enable_excel_processing=False,  # 需要Windows环境时设为True
    enable_data_import=False  # 需要实际数据时设为True
)

print("\n执行报告:")
print(f"状态: {full_report['pipeline_status']}")
print(f"开始时间: {full_report['start_time']}")
print(f"结束时间: {full_report['end_time']}")
print(f"耗时: {full_report['duration_seconds']:.2f} 秒")

## 7.8 使用说明

In [ ]:
print("""
================================================================
                    财务指标导入系统使用说明
================================================================

1. 环境配置
   - 安装Python 3.8+
   - 安装依赖: pip install -r requirements.txt
   - 配置环境变量（数据库密码、同花顺账号等）

2. 同花顺API提取
   - 安装同花顺iFinD终端
   - 配置API账号环境变量
   - 设置日期列表

3. Excel处理（Windows环境）
   - 安装Microsoft Excel
   - 安装pywin32: pip install pywin32
   - 准备Excel文件（文件名格式：YYYYMMDD.xlsx）

4. 数据导入
   - 确保PostgreSQL数据库可访问
   - 目标表会自动创建

5. 执行方式
   - 单步执行：调用相应的函数
   - 完整流程：调用run_full_pipeline()

================================================================
""")